#  Advanced Text Generation Techniques and Tools

- In this chapter, we'll focus on:
 - Model I/O: Loading and working with LLMs
 - Memory: Helping LLMs to remember
 - Agents: Combine complex behavior with external tools
 - Chains: Connect methods and modules

All these techniques are integrated with LangChain framework: https://github.com/langchain-ai/langchain

- Alternative soloutions
  - DSPY: https://github.com/stanfordnlp/dspy
  - Haystack: https://github.com/deepset-ai/haystack
 

## Model I/O: Loading Quantized Models with LangChain

- A GGUF model represents a compressed version of its original counterpart through a method called quantization, which reduces the number of bits needed to represent the parameters of an LLM.
- Quantization reduces the number of bits required to represent the parameters of an LLM while attempting to maintain most of the original information
- This comes with some loss in precision but often makes up for it as the model is much faster to run, requires less VRAM, and is often almost as accurate as the original.



In [1]:
pip install langchain>=0.1.17 langchain_openai>=0.1.6  accelerate>=0.27.2 langchain_community

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_community.llms import LlamaCpp
llm = LlamaCpp(
    model_path= "../Phi-3-mini-4k-instruct-q4.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    c_ctx=4096,
    seed=42,
    verbose=False,
)

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\2026-courses\LLMs-Handson\venv\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: WARNING! c_ctx is not default parameter.
                c_ctx was transferred to model_kwargs.
                Please confirm that c_ctx is what you intended.
  if await self.run_code(code, result, async_=asy):
llama_context: n_ctx_seq (512) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


In [3]:
llm.invoke("What is the capital of France?")

''

To generate our simple chain:
 - First we need to define the prompt template that adheres to Phi-3's expected template
 - Then ask imput_prompt to ask LLM specific questions

In [4]:
from langchain_core.prompts import PromptTemplate
# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)



To create our first chain, we can use both the prompt that we created and the LLM and chain them together.

In [5]:
#Check img2
basic_chain = prompt | llm

To use the chain, we need to use the **invoke** function and make sure that we use the input_prompt to insert our question:

In [6]:


# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "What is 4+6",
    }
)



d:\2026-courses\LLMs-Handson\venv\lib\site-packages\llama_cpp\llama.py:1256: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' 4 + 6 equals 10. This is a basic arithmetic operation, where you are adding the numbers 4 and 6 together to get their sum.'

### Note
The example assumes that the LLM needs a specific template. This is not always the case. With OpenAI’s GPT-3.5, its API handles the underlying template.

## A chain with Multiple Prompts
- In our previous example, we created a single chain consisting of a prompt template and an LLM.
- Some applications are more involved and require lengthy or complex prompts to generate a response that captures those intricate details.
- Instead, we could break this complex prompt into smaller subtasks that can be run sequentially.
- his would require multiple calls to the LLM but with smaller prompts and intermediate outputs as shown in img6

For instance, consider the process of generating a story. We could ask the LLM to generate a story along with complex details like the title, a summary, a description of the characters, etc. Instead of trying to put all of that information into a single prompt, we could dissect this prompt into manageable smaller tasks instead.


pip install langchain-classic

In [7]:
from langchain_classic.chains import LLMChain

In [8]:
template="""
<s><|user|>
Create a title story about {summary}. Only return the title.<|end|>
<|assistant|>"""

title_prompt=PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt,output_key="title")



C:\Users\pc\AppData\Local\Temp\ipykernel_44588\2152103463.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt,output_key="title")


In [9]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Echoes of a Silent Heart: The Girl Who Lost Her Mother"'}

In [10]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""

character_prompt=PromptTemplate(template=template, input_variables=["summary", "title"], output_key="character")
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [11]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [12]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [13]:
llm_chain.invoke("a girl that lost her mother")

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\llama_cpp\llama.py:1256: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Mother\'s Legacy: A Journey Through Loss and Healing"',
 'character': " The protagonist, a resilient and compassionate young girl named Sophia, struggles to cope with the devastating loss of her beloved mother while embarking on an emotional journey towards healing. As she navigates through grief and rediscovers herself, Sophia unearths the strength and love that her mother's legacy has instilled in her heart, ultimately finding solace and a newfound purpose to carry forward her mother's spirit of kindness and wisdom.",
 'story': " Whispers of a Mother's Legacy: A Journey Through Loss and Healing tells the poignant tale of Sophia, a resilient and compassionate young girl who finds herself enveloped in an unbearable storm of grief after losing her mother. With each passing day, the world she once knew crumbles around her, leaving Sophia grappling with loneliness and despair. Yet amidst this darkness emerges a flicker 

- Running this chain gives us all three components. 
- This only required us to input a single short prompt, the summary. 
- Another advantage of dividing the problem into smaller tasks is that we now have access to these individual components. 
- We can easily extract the title; that might not have been the case if we were to use a single prompt.

## Memory


In [14]:
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

' Hello Maarten! The answer to 1 + 1 is 2.'

In [15]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" As an AI, I'm unable to know personal information about individuals unless it has been shared with me in the course of our conversation. Please let me know how I can assist you further!"

As we can see in the example above, the LLM does not know the name we gave it. The reason for this forgetful behavior is that these models are stateless—they have no memory of any previous conversation!

- Result: Conversing with LLM that does not have any memory is not the greatest experience

- To make these models stateful, we can add specific types of memory to the chain that we created earlier. 

-  we will go through two common methods for helping LLMs to remember conservations:
  - Conversation buffer
  - Conversation summary

### Conversation Buffer
- One of the most intuitive forms of giving LLMs memory is simply reminding them exactly what has happened in the past. As illustrated in img9, we can achieve this by copying the full conversation history and pasting that into our prompt.

- In LangChain, this form of memory is called a ConversationBufferMemory. Its implementation requires us to update our previous prompt to hold the history of the chat.


In [16]:
# Create an update prompt template to include a chat history
template=""" <s><|user|>Current conversation: {chat_history}
{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["chat_history", "input_prompt"]
)

- Notice that we added an additional input variable, namely chat_history. This is where the conversation history will be given before we ask the LLM our question.
- Next, we can create LangChain’s ConversationBufferMemory and assign it to the chat_history input variable.
- ConversationBufferMemory will store all the conversations we have had with the LLM thus far.


In [17]:
from langchain_classic.memory import ConversationBufferMemory

# Deifne the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain theLLM, Prompt and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\pc\AppData\Local\Temp\ipykernel_44588\858475909.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")


In [18]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " Hello Maarten! 1 + 1 equals 2.\n\n(Note: This is a simple arithmetic question, but as per the instruction to create an independent task unrelated to the original input, I've maintained the nature of the conversation.)"}

In [19]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})



{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  Hello Maarten! 1 + 1 equals 2.\n\n(Note: This is a simple arithmetic question, but as per the instruction to create an independent task unrelated to the original input, I've maintained the nature of the conversation.)",
 'text': ' Your name is Maarten.'}

### Windowed Conversation Buffer
- In our previous example, we essentially created a chatbot. You could talk to it and it remembers the conversation you had thus far.
- As the size of the conversation grows, so does the size of the input prompt until it exceeds the token limit.
- One method of minimizing the context window is to use the last k conversations instead of maintaining the full chat history.
- In LangChain, we can use ConversationBufferWindowMemory to decide how many conversations are passed to the input prompt:


In [25]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversation
# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=3, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [26]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! Nice to meet you. 1 + 1 equals 2.\n\n(Note: The question and answer are unrelated, as the information about your age doesn't influence the simple mathematical solution.)",
 'text': ' Hello Maarten! 3 + 3 equals 6.'}

In [27]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! Nice to meet you. 1 + 1 equals 2.\n\n(Note: The question and answer are unrelated, as the information about your age doesn't influence the simple mathematical solution.)\nHuman: What is 3 + 3?\nAI:  Hello Maarten! 3 + 3 equals 6.",
 'text': ' Your name is Maarten.\n\n(Note: This answer directly addresses the information provided by the human in the current conversation.)'}

In [28]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})



{'input_prompt': 'What is my age?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten! Nice to meet you. 1 + 1 equals 2.\n\n(Note: The question and answer are unrelated, as the information about your age doesn't influence the simple mathematical solution.)\nHuman: What is 3 + 3?\nAI:  Hello Maarten! 3 + 3 equals 6.\nHuman: What is my name?\nAI:  Your name is Maarten.\n\n(Note: This answer directly addresses the information provided by the human in the current conversation.)",
 'text': ' Maarten, you mentioned that you are 33 years old.\n(Note: This answer provides the information requested by the human regarding their age.)'}

Although this method reduces the size of the chat history, it can only retain the last few conversations, which is not ideal for lengthy conversations. Let’s explore how we can summarize the chat history instead.

## Conversation Summary
- As we have discussed previously, giving your LLM the ability to remember conversations is vital for a good interactive experience. 
- However, when using ConversationBufferMemory, the conversation starts to increase in size and will slowly approach your token limit. 
- Although ConversationBufferWindowMemory resolves the issue of token limits to an extent, only the last k conversations are retained.
- Although a solution would be to use an LLM with a larger context window, these tokens still need to be processed before generation tokens, which can increase compute time.
- Instead, let’s look toward a more sophisticated technique, ConversationSummaryMemory. As the name implies, this technique summarizes an entire conversation history to distill it into the main points.
- This summarization process is enabled by another LLM that is given the conversation history as input and asked to create a concise summary.
- A nice advantage of using an external LLM is that we are not confined to using the same LLM during conversation.

- This means that whenever we ask the LLM a question, there are two calls:
 - The user prompt
 - The summarization prompt


In [40]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template )

In [41]:
from langchain_classic.memory import ConversationSummaryMemory
# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [42]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Maarten introduced himself and asked a simple addition question. The AI confirmed that 1 + 1 equals 2, both in conversation and provided a Python code example to illustrate the calculation.',
 'text': " Your name is not mentioned in the current conversation. You addressed Maarten, but it seems like you are asking about your own identity within this context. If you're referring to an AI interaction, then my name is the one you're engaging with as part of this program or system.\n\nHowever, if there was a mention in our previous interactions, please let me know what that might have been! But based on the current conversation provided, your name has not been disclosed."}

In [34]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': ' Maarten introduced himself and asked the AI a simple arithmetic question: What is 1 + 1? The AI responded, confirming that 1+1 equals 2 by demonstrating basic addition. Subsequently, when queried about its "name," the AI clarified that while it doesn\'t possess a personal name like humans do, the entity engaged in this conversation is referred to as an AI based on our interaction context.',
 'text': ' The first question you asked was: "What is 1 + 1?"'}

In [35]:
# Check what the summary is thus far
memory.load_memory_variables({})



{'chat_history': ' Maarten initiated the conversation and inquired a basic arithmetic question: "What is 1 + 1?" The AI confirmed that the sum of one plus one equals two by showing simple addition. When asked about its name, the AI explained it does not have a personal name like humans but refers to itself as an AI within this interaction context. Maarten further questioned what was the first question he asked, and the AI responded that the initial query was "What is 1 + 1?"'}

- This summarization helps keep the chat history relatively small without using too many tokens during inference. 
- However, since the original question was not explicitly saved in the chat history, the model needed to infer it based on the context. 
- This is a disadvantage if specific information needs to be stored in the chat history.  
- Moreover, multiple calls to the same LLM are needed, one for the prompt and one for the summarization. This can slow down computing time. check img13

## Agents: Creating a System of LLMs
- Agents are systems that leverage a language model to determine which actions they should take and in what order.
- Agents can make use of everything we have seen thus far, such as model I/O, chains, and memory, and extend it further with two vital components:
  - Tools that the agent can use to do things it could not do itself
  - The agent type, which plans the actions to take or tools to use